# 05 · Analysis and Reporting

Reproduce the demonstration analysis or supply JSON exports for a single mode. Upload the project ZIP when prompted. The default data are synthetic.

In [ ]:
import io
import json
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "check-before-you-run")
             if (p / "src" / "study.py").is_file()), None)
if root is None:
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Upload the original project ZIP.")
    with ZipFile(io.BytesIO(next(iter(uploads.values())))) as archive:
        paths = [PurePosixPath(name) for name in archive.namelist()]
        if not paths or any(path.is_absolute() or ".." in path.parts for path in paths):
            raise ValueError("Invalid project archive.")
        archive.extractall(Path.cwd())
    root = Path.cwd() / "check-before-you-run"
    if not (root / "src" / "study.py").is_file():
        raise ValueError("Use the original check-before-you-run.zip.")
root = root.resolve()
sys.path.insert(0, str(root))
from src import study

print("Project ready")

In [ ]:
mode = "synthetic"
response_folder = Path.cwd() / "private_records"
upload_responses = False

if mode == "synthetic":
    records = json.loads((root / "demo" / "synthetic_records.json").read_text())
elif upload_responses:
    from google.colab import files
    records = [json.loads(data) for data in files.upload().values()]
else:
    records = [json.loads(path.read_text()) for path in sorted(response_folder.glob("*.json"))]
rows = study.score_sessions(records, mode)
print(f"{len(records)} {mode} sessions | {len(rows)} scored rounds")

In [ ]:
from collections import Counter
from statistics import mean

outcomes = ("valid_counterexample", "discriminating_input", "correct_expected_output",
            "practice_discriminating_input", "practice_verdict_correct")
print("Sequence counts:", dict(sorted(Counter(record["sequence"] for record in records).items())))
paired = {}
for condition in study.CONDITIONS:
    selected = [row for row in rows if row["condition"] == condition]
    for metric in outcomes:
        print(f"{condition} | {metric}: {sum(row[metric] for row in selected)}/{len(selected)}")
    paired[condition] = {
        participant: mean(row["valid_counterexample"] for row in selected if row["participant"] == participant)
        for participant in sorted({row["participant"] for row in selected})
    }
differences = [paired["B"][key] - paired["A"][key] for key in paired["A"]]
print(f"Mean paired difference (B minus A): {100 * mean(differences):+.1f} percentage points")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
pairs = Counter((paired["A"][key], paired["B"][key]) for key in paired["A"])
for (a, b), count in sorted(pairs.items()):
    ax.plot([0, 1], [a, b], color="#214b98", alpha=0.60, linewidth=1 + count)
for column, condition in enumerate(("A", "B")):
    frequencies = Counter(paired[condition].values())
    ax.scatter([column] * len(frequencies), list(frequencies),
               s=[35 + 32 * count for count in frequencies.values()], color="#214b98")
ax.set(xlim=(-0.3, 1.3), ylim=(-0.08, 1.08), yticks=[0, 0.5, 1],
       ylabel="Share of valid independent tests",
       title=f"{mode.capitalize()} data · {len(records)} paired sessions")
ax.set_xticks([0, 1], ["Run first", "Predict first"])
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
out = Path.cwd() / "results"
out.mkdir(exist_ok=True)
fig.savefig(out / f"{mode}-paired-results.png", dpi=170)
plt.show()